## tiiuae/Falcon3-1B-Instruct

In [2]:
import evaluate
import json
import os
import pandas as pd
import torch

from datasets import Dataset
from pathlib import Path
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer


### DIRECTORIES

In [3]:
PROJECT_PATH = Path(os.getcwd()).parent.parent
DS_PATH = PROJECT_PATH / "datasets" / "fine-tuning" / "conversational"
TRAINING_DS_PATH = DS_PATH / "ordered_training_ds_juliet.jsonl"
TESTING_DS_PATH = DS_PATH / "ordered_testing_ds_juliet.jsonl"

LOGS_PATH = PROJECT_PATH / "logs" / "falcon3-1b-instruct"
OUTPUT_METRICS_PATH = PROJECT_PATH / "output_metrics"
OUTPUT_METRICS_FILE = OUTPUT_METRICS_PATH / "falcon3-1B-instruct.txt"

MODEL_OUTPUT_PATH = PROJECT_PATH / "fine-tuned-models" / "falcon3-1B-instruct"

LOGS_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_METRICS_PATH.mkdir(exist_ok=True)
MODEL_OUTPUT_PATH.mkdir(exist_ok=True)

### PARAMETERS

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "tiiuae/Falcon3-1B-Instruct"

### TRAINING AND TESTING DATASET

In [5]:
train_data = []
with open(TRAINING_DS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        train_data.append(json.loads(line.strip()))

test_data = []
with open(TESTING_DS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        test_data.append(json.loads(line.strip()))

first_15K_training_examples = train_data[:15000]
first_5k_eval_examples = test_data[:5000]

### Refine dataset to Input - Label

In [6]:
training_dataframe = pd.DataFrame(first_15K_training_examples, columns=["messages", "cwe", "code", "filename", "type"])[["messages"]]

### Get Model and Tokenizer

In [7]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float8_e5m2,
    bnb_4bit_use_double_quant=True,
)

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    trust_remote_code=True,
    device_map="auto"
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [9]:
if tokenizer.chat_template is not None:
    print(tokenizer.chat_template)
else:
    print("No chat template")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

{%- if tools %}
{{- '<|system|>\n' }}
{%- if messages[0]['role'] == 'system' %}
{{- messages[0]['content'] }}
{%- set remaining_messages = messages[1:] %}
{%- else %}
{%- set remaining_messages = messages %}
{%- endif %}
{{- 'You are a Falcon assistant skilled in function calling. You are helpful, respectful, and concise.\n\n# Tools\n\nYou have access to the following functions. You MUST use them to answer questions when needed. For each function call, you MUST return a JSON object inside <tool_call></tool_call> tags.\n\n<tools>' + tools|tojson(indent=2) + '</tools>\n\n# Output Format\n\nYour response MUST follow this format when making function calls:\n<tool_call>\n[\n  {"name": "function_name", "arguments": {"arg1": "value1", "arg2": "value2"}},\n  {"name": "another_function", "arguments": {"arg": "value"}}\n]\n</tool_call>\nIf no function calls are needed, respond normally without the tool_call tags.\n' }}
{%- for message in remaining_messages %}
{%- if message['role'] == 'user' %}


In [10]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(131072, 2048)
    (layers): ModuleList(
      (0-17): 18 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNo

### Tokenize Datasets

In [11]:
def tokenize_message(example):
    return tokenizer.apply_chat_template(example, tokenize=False)

training_dataset = Dataset.from_pandas(training_dataframe)

### Accuracy Compute

In [12]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    return {}

### LoRA Config

In [13]:
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

In [14]:
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

### TRAINING PARAMETERS

In [15]:
BATCH_SIZE = 2
LEARNING_RATE = 2e-5
LOGGING_STEPS = 200
GRADIENT_ACCUMULATION = 4
MAX_STEPS = 1875
WARMUP_RATIO = 0.1

In [16]:
training_args = SFTConfig(
    output_dir=MODEL_OUTPUT_PATH.resolve(),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    logging_dir=str(LOGS_PATH.resolve()),
    logging_steps=LOGGING_STEPS,
    max_steps=MAX_STEPS,
    fp16=True,
    learning_rate=LEARNING_RATE,
    save_total_limit=1,
    gradient_checkpointing=True,
    push_to_hub=False,
    eval_accumulation_steps=1,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [17]:
trainer = SFTTrainer(
    model=model,
    train_dataset=training_dataset,
    args=training_args,
    peft_config=peft_config,
)

C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\peft\tuners\lora\bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Tokenizing train dataset:   0%|          | 0/15000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/15000 [00:00<?, ? examples/s]

In [18]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 2023}.
C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
200,2.106018
400,0.955306
600,0.465942
800,0.356650
1000,0.299657
1200,0.264623
1400,0.246350
1600,0.229380
1800,0.227772


C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:\Users\pablo\Projects\infosec-master-thesis\src\llm\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences

TrainOutput(global_step=1875, training_loss=0.5582148101806641, metrics={'train_runtime': 19055.2823, 'train_samples_per_second': 0.787, 'train_steps_per_second': 0.098, 'total_flos': 9.920940074532864e+16, 'train_loss': 0.5582148101806641, 'entropy': 0.2745779252052307, 'num_tokens': 10654343.0, 'mean_token_accuracy': 0.9494708371162415, 'epoch': 1.0})

In [19]:
save_file_path = Path("~/ft/falcon3-1B-instruct-fine-tuned").expanduser()
trainer.model.save_pretrained(save_directory=save_file_path.resolve())